In [79]:
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


In [80]:
ds = pd.read_csv('data-a1.csv')
ds_copy = ds.copy()
print(ds.info()) # drop rows with missing values in engine column, since there are only 2 missing values, this will not affect our dataset much


<class 'pandas.DataFrame'>
RangeIndex: 62302 entries, 0 to 62301
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   addref        62302 non-null  int64  
 1   city          62302 non-null  str    
 2   assembly      19374 non-null  str    
 3   body          55211 non-null  str    
 4   make          62302 non-null  str    
 5   model         62302 non-null  str    
 6   year          58516 non-null  float64
 7   engine        62299 non-null  float64
 8   transmission  62302 non-null  str    
 9   fuel          61593 non-null  str    
 10  color         61110 non-null  str    
 11  registered    62302 non-null  str    
 12  mileage       62302 non-null  int64  
 13  price         61830 non-null  float64
dtypes: float64(3), int64(2), str(9)
memory usage: 6.7 MB
None


In [81]:
# print(ds.head(5))# 0 missing values
print(f'null fuel {ds["fuel"].isnull().sum()}')  # 2 missing values
print(f'null transmission {ds["transmission"].isnull().sum()}')  # 2 missing values
print(f'null assembly {ds["assembly"].isnull().sum()}')  # 0 missing values   
print(f'null city {ds["city"].isnull().sum()}')  # 0 missing values
print(f'null registered {ds["registered"].isnull().sum()}')  # 0 missing values
print(f'null body {ds["body"].isnull().sum()}')  # 0 missing values
print(f'null color {ds["color"].isnull().sum()}')  # 0 missing values
print(f'null model {ds["model"].isnull().sum()}')  # for make we must use target encoding, since there are several unique values, one hot encoding will create 6 columns, which is too much for our dataset
print(f'null year {ds["year"].isnull().sum()}')  
print(f'null engine {ds["engine"].isnull().sum()}')  # for fuel we can use one hot encoding, since there are only 3 unique values, one hot encoding will create 3 columns, which is not too much for our dataset

null fuel 709
null transmission 0
null assembly 42928
null city 0
null registered 0
null body 7091
null color 1192
null model 0
null year 3786
null engine 3


In [ ]:

 # drop rows with missing values in engine column, since there are only 2 missing values, this will not affect our dataset much
ds = ds.dropna(subset=['engine']) 
print(f'null engine {ds["engine"].isnull().sum()}')

null engine 0


In [ ]:
print(ds['price'].isnull().sum())  # 0 missing values
ds.dropna(subset=['price'], inplace=True)
print(ds['price'].isnull().sum())


472
0


In [93]:
col_with_missing_values = ds.isnull().any()
print(col_with_missing_values[col_with_missing_values == True])

assembly    True
body        True
year        True
color       True
car_age     True
dtype: bool


In [94]:
missing_percent = ds.isnull().mean() * 100
print(missing_percent[missing_percent > 0])

ds['year'] = ds.groupby(['make', 'model'])['year'].transform(lambda x: x.fillna(x.median()))
# print(ds['year'])

ds['car_age'] = 2026 - ds['year']
# print(ds['car_age'])

ds['engine'] = ds.groupby(['make', 'model'])['engine'].transform(lambda x: x.fillna(x.median()))
# print(ds['engine'])


assembly    69.041034
body        11.412490
year         0.118071
color        1.851942
car_age      0.118071
dtype: float64


In [95]:

ds['fuel'] = ds['fuel'].fillna(ds['fuel'].mode()[0])
ds['transmission'] = ds['transmission'].fillna(ds['transmission'].mode()[0])

cols_to_encode = ['transmission', 'fuel', 'assembly']
ds_encoded = pd.get_dummies(ds, columns=cols_to_encode, drop_first=True)


In [100]:
print(f'unique cities: {ds["city"].nunique()}') # for year we will use target encoding, since there are several unique values, one hot encoding will create 6 columns, which is too much for our dataset
print(f'unique registered: {ds["registered"].nunique()}') # for year we will use target encoding, since there are several unique values, one hot encoding will create 6 columns, which is too much for our dataset
print(f'unique body: {ds["body"].nunique()}') # for body we must use target encoding, since there are several unique values, one hot encoding will create 6 columns, which is too much for our dataset
print(f'unique color: {ds["color"].nunique()}')  # for color we must use target encoding, since there are several unique values, one hot encoding will create 6 columns, which is too much for our dataset
print(f'unique make: {ds["make"].nunique()}')  # for make we must use target encoding, since there are several unique values, one hot encoding will create 6 columns, which is too much for our dataset
print(f'unique model: {ds["model"].nunique()}')  # for make we must use target encoding, since there are several unique values, one hot encoding will create 6 columns, which is too much for our dataset


unique cities: 287
unique registered: 114
unique body: 21
unique color: 373
unique make: 66
unique model: 405


In [123]:

# encoding 'registered' column by collapsing rare categories into 'Other'

min_count = 1000
protected = ['Un-Registered']
counts = ds['registered'].value_counts()
# print(counts.head(10))

rare_categories = counts[(counts < min_count) & ((counts.index.isin(protected)) == False)].index
# print(rare_categories)

ds['registered_collapsed'] = ds['registered'].where(
    ds['registered'].isin(rare_categories) == False, 'Other'
)

# print(ds['registered_collapsed'].value_counts())

ds = pd.get_dummies(ds, columns=['registered_collapsed'], prefix='registered', drop_first=True)

# print(ds_encoded.head(5))

In [ ]:

cols_to_encode = ['make', 'model']
target_encoder = TargetEncoder(cols=cols_to_encode, smoothing=10, handle_missing='value', handle_unknown='value')
ds_encoded = target_encoder.fit_transform(ds_encoded, ds_encoded['price'])

print(ds_encoded.head(5))



unique cities: 287
unique registered: 114
unique body: 21
unique color: 373
unique make: 66
unique model: 405


TypeError: bad operand type for unary ~: 'str'

In [ ]:
X = ds.iloc[:, 0:-1].values
y = ds.iloc[:, -1].values
print(ds.head(5))